## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


By using nf-core pipelines, we get the same results folders and files between different runs. Hence, the data is reproducible, because we have the same pipelines with the same output formats and well documented processes and steps. Additionally, we have better control over tool versions, which also plays a big part in reproducability.

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

We have to performe sequence alignment, which we can perform via nf-core rnaseq (url:"https://nf-co.re/rnaseq/3.27.0", date: 29.09.2026). This pipeline, which is an RNA sequencing analysis pipeline, performs quality control in it's 1st pre-processing step via the FastQC (and also after trimming). Hence, we can use this to also check our fastq file quality.

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

The rnaseq pipeline is a RNA sequencing pipeline. In short, the pipeline takes fastq reads and alignes them to a given reference genome, resulting in a count table. rnaseq does this by utilizing these of 5 main steps:
1. Pre-processing
2. Genome alignment & quantification
3. Pseudo-alignment & quantification
4. Post-processing
5. Quality control & reporting

Main tools in the pipeline are FastQC in the first, STAR, Bowtie2 and HISAT2 aligner in the second, Salmon or Kallisto counters in the third, and MultiQC in the third and fifth step of the pipeline.

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

I reuse the samplesheet from the earlier fetchngs pipeline run. But for the samplesheet to work, I had to add 'strandedness' column and rename the 'instrument_platform' into 'seq_platform'. Because this was not much work, I did this manually. Then I can use the samplesheet as input for the rnaseq pipeline. The rest of the sheet stays the same.

In [ ]:
# post here the command you used to run nf-core/rnaseq
!NXF_VER=25.10.4 nextflow run nf-core/rnaseq -r 3.27.0 -profile podman \
                                             --input SRA_runs/samplesheet/samplesheet.csv \
                                             --fasta mus_musculus_ref/ncbi_dataset/data/GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.fna \
                                             --gff mus_musculus_ref/ncbi_dataset/data/GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.gff \
                                             --skip_alignment \
                                             --skip_pseudo_alignment \
                                             --outdir ./rnaseq \

# Note: I explicitly used nextflow version 25.10.4, because some parameters did not not work with the newest version. Alternatively to changing the version, I could've also used a param sheet, defining parameters in there. But because my run worked and it took long, I didn't redo it.

Explain all the parameters you set and why you set them in this way.



Parameters I set:
1. --input: is set to the modified fetchngs samplesheet, because it contains all the necessary metadata for the 2 fastq files I want to process
2. --fasta: is set to a mus musculus reference genome fasta file. This is because the pipeline needs a reference fasta to align the input reads onto for an alignment. Because we look at mus musculus reads, we of course need the same reference.
3. --gtf: is set to a mus musculus gtf file, which contains the necessary gene information to count gene abundance. This gtf matches the fasta reference file.
4. --skip_(pseudo)_alignment: is set to skip the main alignment step. This is done, to fasten up processing. Else, if I would want the count table, I would need the alignment. And yes, even though I skipped the alignment, I set the fasta and gtf just for safety to reduce possible error sources.

## Browsing the results

How did the pipeline perform?

Note: For the following tasks I will reference the results pushed by the tutors with all SRA runs and not only the two smallest ones.

In regards of pipeline performance, the pipeline for me took at minimum half an hour without the alignment step and only the two specified, smallest SRAs. Hence, with alignment or even more reads, like the tutors solution, I would assume that it would need even much, much longer. I would approximate at least two hours.

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication 
- GC content

What are the possible steps that could lead to poorer results?

FastQC: performs quality on fastq files and generates an html report (performed for raw and trimmed reads)
QualiMap:  performs quality control on alignment data
RseQC: python library for QC on RNA sequences
dupRadar: performs duplication rate quality control for dataset

Overall, I would say, I am satisfied with the QC steps and quality of the samples. I would only see one outlier over all samples, who doesn't seem to be of good quality.

- GC content: could indicate possible contamination of the sample, if the GC content is higher/lower than normally. And this, of course, could then affect the read quality.
- Ribosomal RNA: high rRNA proportion can indicate incomplete rRNA depletion in pre-processing. This might affect further gene expression analysis, if the main focus is not the rRNA.
- Duplication: Could indicate low library complexity or PCR amplification base, which might affect quality, but mustn't.

specific steps that could lead to poorer results:
- trimming -> to much trimming could erase important base information. Not enough trimming on the other hand, could leave adapter sequences
- alignment: If references are not good enough, this step could produce bad quality alignment (also heavily dependent on tool)
- BBSplit/SortMeRNA: poor referencesequences/settings could lead erronious deletion of reads
BUT generally: if overall reference quality is bad or tool (all tools) specific settings are not fullfilled/ applied falsly, then every step could contain possible error sources.

Would you exclude any samples? If yes, which and why?

If I had to exclude samples, I would exclude SRX19144492, because of the above mentioned high GC contents.


What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

I would perform differential gene expression analysis of the resulting count table data for the different coditions and genotypes (maybe use the differential abundance nf-core pipeline). This is, because the authors try to answer the question what effect Oxycodon and/or SNI have on the gene expression. For differential gene expression analysis, one could probably use the deSeq2 package in R (if not nf-core).